-przewidujemy stack voltage traktujac reszte elementow jako inputy: 
-reszta kolumn to x 
-kolumna 'szafa' pozostaje ignorowana
-model dla kazdego compound przeprowadzony oddzielnie

In [123]:
#import bibliotek
import pandas as pd
import numpy as np
import warnings 

#import elementow do modelu sieic: 
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import joblib
#do zapisu modelu na dysku: 
import re 

In [124]:
#odczyt danych z pliku csv
df= pd.read_csv('data_csv.csv')
#celem do przewidzenia jest 'Stack voltage'
target = 'Stack voltage'
#usuniecie wierszy bez targetu:
df = df.dropna(subset = [target]).copy()
# df.head()

In [125]:
#przeglad kontrolny ilsoci wierszy i kolumn: 
# print(f"liczba wierszy to: {len(df)}")
# print(f"uzyte kolumny: {df.columns}")

In [126]:
#unifkacja i sprawdzenie danych: 
df.columns = df.columns.str.strip().str.lower()
# df.head()

In [132]:
#ustawienie nazw kolumn: 
target = 'stack voltage'
columns_ignored = ['szafa','compound',target]

#jezeli nie ma kolumny docelowej, to wyrzucamy blad:
if target not in df.columns:
    raise ValueError(f"Nie można znaleźć kolumny docelowej: {target}")

cols = []

for c in ['compound','szafa',target]:
    if c in df.columns: 
        cols.append(c)
# df[cols].head()


In [141]:
#lista badanych srodkow: (unikalne i pozbyte pustych wartosci)
compounds = df['compound'].dropna().unique()
print(f"ilosc unikalnych copounds to {len(compounds)},\n a sa to: {compounds}")   

ilosc unikalnych copounds to 5,
 a sa to: ['H2, 605C' 'MeOH+H2O' 'MeOH+H2O, 605C ' 'MeOH+H2O, 605C' 'MeOH+H2O, 640C']


In [134]:
def prepare_data(df,compound,columns_ignored,target):
    #wiersze tylko dla danego compoundu:
    print("____________________________________")
    print(f"generacja dla compound: {compound}")
    d_compound = df[df['compound'] == compound].copy()
    # print(d_compound.head(5))
    

    #tworzenie listy inputow:
    x = d_compound.drop(columns = columns_ignored)
    
    #jezeli jakas kolukmna x jest pusta to ja zrzucamy:
    empty_x = []
    for col in x.columns:
        if x[col].isnull().all():
            empty_x.append(col)
    if len(empty_x) > 0:
        print(f"usuwam puste kolumny x: {empty_x}")
        x = x.drop(columns=empty_x)
        
    # print(x.head(5))

    #kolumna y:
    y = d_compound[target]
    # print(y.head(5))

    #zabezpieczenie zeby wszytko bylo liczba:
    x = x.apply(pd.to_numeric, errors='coerce')
    y = pd.to_numeric(y, errors='coerce')

    #jezeli wystepuje choc pare wartosci bez liczb to je zrzucamy:
    mask_number = x.notna().all(axis=1) & y.notna()

    #zostawiamy wiersze bez brakujacych danych:
    x= x[mask_number]
    
    print(f"ilosc x po usunieciu brakujacych danych: {len(x)}")
    y = y[mask_number]
    print(f"ilosc y po usunieciu brakujacych danych: {len(y)}")
    
    #zwort dannych:
    return x,y
    # # jezeli gdzies nie ma liczby zapisac i wyswietlic blad:
    # if x.isnull().values.any() or y.isnull().values.any():
    #     raise ValueError("Dane zawierają wartości, które nie są liczbami. Proszę sprawdzić dane wejściowe.")



In [137]:
x,y = prepare_data(compound=compounds[3],df=df,columns_ignored=columns_ignored,target=target)

____________________________________
generacja dla compound: MeOH+H2O, 605C
usuwam puste kolumny x: ['co2.1', 'h2o']
ilosc x po usunieciu brakujacych danych: 50
ilosc y po usunieciu brakujacych danych: 50


In [138]:
#trening jednego modelu:
def modeling(comp,x,y):
    

    #niedostateczny rozmiar danych:
    if len(x) < 20:
        print(f"Za mało danych dla compound {comp}: {len(x)} wierszy. Pomijam ten compound.")
        return None,None,None
    else:
        #podzial danych na treningowe i testowe:
        x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

        model = Pipeline(steps = [
            ('scalar', StandardScaler()),
            ('mlp', MLPRegressor(
                hidden_layer_sizes=(32, 16),
                activation = 'relu',
                solver = 'adam',
                max_iter = 2000, #liczba interacji!!!!
                random_state=42 
            ))
        ])

        model.fit(x_train, y_train)
        y_pred = model.predict(x_test)

        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        
        
        print(f"Modelowanie dla [{comp}]:")
        print(f"Test MAE dla [{comp}]:", mae)
        print(f"Test RMSE dla [{comp}]:", rmse)
        print("____________________________________")
        
        return mae,rmse,model



In [142]:
#trening wszyskich modeli:
results = {}
models = {}

for comp in compounds:
    x,y = prepare_data(df,comp,columns_ignored,target)
    mae,rmse,model  = modeling(comp,x,y)

    if model is not None:
        models[comp] = model
        results[comp] = {"rows": len(x), "mae": mae, "rmse": rmse, "status": "OK"}
    else:
        results[comp] = {"rows": len(x), "mae": None, "rmse": None, "status": "SKIPPED"}
    

# wyniki w tabelce
results_df = pd.DataFrame(results).T.sort_values(["status", "rows"], ascending=[True, False])
results_df.to_csv("results.csv")

____________________________________
generacja dla compound: H2, 605C
usuwam puste kolumny x: ['h2o', 'ch3oh']
ilosc x po usunieciu brakujacych danych: 62
ilosc y po usunieciu brakujacych danych: 62
Modelowanie dla [H2, 605C]:
Test MAE dla [H2, 605C]: 0.15813470729481122
Test RMSE dla [H2, 605C]: 0.2070604021534149
____________________________________
____________________________________
generacja dla compound: MeOH+H2O
usuwam puste kolumny x: ['co2.1', 'h2o']
ilosc x po usunieciu brakujacych danych: 31
ilosc y po usunieciu brakujacych danych: 31
Modelowanie dla [MeOH+H2O]:
Test MAE dla [MeOH+H2O]: 0.08217441461801744
Test RMSE dla [MeOH+H2O]: 0.09933302018301247
____________________________________
____________________________________
generacja dla compound: MeOH+H2O, 605C 
usuwam puste kolumny x: ['co2.1', 'h2o']
ilosc x po usunieciu brakujacych danych: 18
ilosc y po usunieciu brakujacych danych: 18
Za mało danych dla compound MeOH+H2O, 605C : 18 wierszy. Pomijam ten compound.
______

In [144]:
#zapis modeli na dysku:

def safe_filename(s):
    s = str(s)
    s = re.sub(r"[^a-zA-Z0-9_\-]+", "_", s)
    return s.strip("_")

for comp, model in models.items():
    name = safe_filename(comp)
    joblib.dump(model, f"model_{name}.joblib")

print("Zapisano modeli:", len(models))

Zapisano modeli: 3


In [145]:
#predykcja dla nowych danych: 
#wybor compoundu:
# wybierz compound, dla którego jest model
comp = next(iter(models.keys()))
model = models[comp]

X, y = prepare_data(df,comp,columns_ignored,target,)

one_row = X.iloc[[0]]  # DataFrame z 1 wierszem
pred_voltage = model.predict(one_row)[0]

print("Compound:", comp)
print("Prawdziwe Stack voltage:", float(y.iloc[0]))
print("Przewidziane Stack voltage:", float(pred_voltage))


____________________________________
generacja dla compound: H2, 605C
usuwam puste kolumny x: ['h2o', 'ch3oh']
ilosc x po usunieciu brakujacych danych: 62
ilosc y po usunieciu brakujacych danych: 62
Compound: H2, 605C
Prawdziwe Stack voltage: 2.67
Przewidziane Stack voltage: 2.6487948071868423
